In [0]:
%sql
CREATE TABLE IF NOT EXISTS rideflow.silver.dim_driver (
  driver_sk      BIGINT GENERATED ALWAYS AS IDENTITY,   -- auto-numbered surrogate key
  driver_id      STRING,
  driver_name    STRING,
  phone          STRING,
  city_id        BIGINT,
  vehicle_type   STRING,
  rating         DOUBLE,
  status         STRING,
  joined_date    DATE,
  row_hash       STRING,
  effective_from TIMESTAMP,
  effective_to   TIMESTAMP,
  is_current     BOOLEAN
)
TBLPROPERTIES (delta.enableChangeDataFeed = true)                    -- record every row change (Change Data Feed)

In [0]:
%sql
SHOW TBLPROPERTIES rideflow.silver.dim_driver

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "rideflow"
TRACKED = ["driver_name", "phone", "city_id", "vehicle_type", "rating", "status"]

def prep_changes(df):
    # a) drop exact duplicate events (same driver + same sequence number)
    df = df.dropDuplicates(["driver_id", "sequence_num"])

    # b) keep only the LATEST event per driver in this batch
    w = Window.partitionBy("driver_id").orderBy(F.col("sequence_num").desc())
    df = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

    # c) types + a hash of the tracked columns
    df = (df.withColumn("change_ts",   F.to_timestamp("change_ts"))
            .withColumn("joined_date", F.to_date("joined_date"))
            .withColumn("row_hash", F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in TRACKED]), 256)))
    return df

In [0]:
changes = prep_changes(spark.read.table(f"{catalog}.bronze.drivers_cdc_raw"))
print(changes.count())      # expect 200
changes.select("driver_id", "city_id", "rating", "row_hash").show(3, truncate=False)

In [0]:
def build_staged(changes):
    current = (spark.read.table(f"{catalog}.silver.dim_driver")
                 .filter("is_current = true")
                 .select("driver_id", F.col("row_hash").alias("cur_hash")))

    # drivers that already exist AND whose values really changed (and weren't deleted)
    changed = (changes.join(current, "driver_id")
                      .filter((F.col("row_hash") != F.col("cur_hash")) & (F.col("operation") != "DELETE"))
                      .drop("cur_hash"))

    staged = (changes.withColumn("merge_key", F.col("driver_id"))                       # copy 1: real key
              .unionByName(changed.withColumn("merge_key", F.lit(None).cast("string"))))  # copy 2: NULL key
    return staged

In [0]:
def apply_scd2(changes):
    build_staged(changes).createOrReplaceTempView("staged")
    spark.sql(f"""
      MERGE INTO {catalog}.silver.dim_driver t
      USING staged s
        ON t.driver_id = s.merge_key AND t.is_current = true

      -- close the current row when the driver changed or was deleted
      WHEN MATCHED AND (s.operation = 'DELETE' OR t.row_hash <> s.row_hash) THEN
        UPDATE SET t.is_current = false, t.effective_to = s.change_ts

      -- open a new row for new drivers and new versions
      WHEN NOT MATCHED AND s.operation <> 'DELETE' THEN
        INSERT (driver_id, driver_name, phone, city_id, vehicle_type, rating, status, joined_date,
                row_hash, effective_from, effective_to, is_current)
        VALUES (s.driver_id, s.driver_name, s.phone, s.city_id, s.vehicle_type, s.rating, s.status, s.joined_date,
        s.row_hash, s.change_ts, NULL, true)
    """)

In [0]:
%sql
SELECT COUNT(*) AS total_rows,
       SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows
FROM rideflow.silver.dim_driver

In [0]:
%sql
SELECT driver_id, city_id, is_current, _change_type, _commit_version
FROM table_changes('rideflow.silver.dim_driver', 1)     -- changes from version 1 onwards
ORDER BY _commit_version, driver_id

In [0]:
def process_driver_batch(batch_df, batch_id):
    changes = prep_changes(batch_df)
    apply_scd2(changes)
    print(f"driver batch {batch_id}: {changes.count()} changes")

chk_root = f"/Volumes/{catalog}/raw/checkpoints"

q = (spark.readStream
        .table(f"{catalog}.bronze.drivers_cdc_raw")
      .writeStream
        .foreachBatch(process_driver_batch)
        .option("checkpointLocation", f"{chk_root}/dim_driver")
        .trigger(availableNow=True)
        .start())
q.awaitTermination()

In [0]:
%sql
SELECT COUNT(*) AS total_rows, SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows
FROM rideflow.silver.dim_driver